# FlashEats — Class 5 Investigation

## Client escalation

> **“Late deliveries are increasing and customers say our ETA is unreliable. Figure out what is happening before we invest in an AI delay-prediction system.”**

Today is not a syntax lesson. Your job is to use SQL, CSV, JSON and APIs like an FDE:

**define the question → retrieve evidence → validate completeness → reconcile sources → state what you know and what you still cannot know**

### Rules
- Do not train an ML model.
- State your definition of **late** before calculating it.
- Preserve raw API responses.
- Do not silently drop failures.
- If another team gets a different answer, investigate the definition and data grain first.

In [1]:
# 0. INSTALLS + IMPORTS
!pip -q install pandas requests flask matplotlib

import os, sys, json, time, sqlite3, subprocess, zipfile
from pathlib import Path

import pandas as pd
import requests
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

print("Environment ready.")

Environment ready.


In [2]:
# 1. LOAD THE CLASSROOM PACK
# Local run: detect local pack folder or current working directory
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif (Path.cwd() / "flasheats" / "database" / "flasheats.db").exists():
    BASE = Path.cwd() / "flasheats"
elif (Path.cwd().parent / "flasheats" / "database" / "flasheats.db").exists():
    BASE = Path.cwd().parent / "flasheats"
elif (Path.cwd() / "flasheats-classroom-pack" / "database" / "flasheats.db").exists():
    BASE = Path.cwd() / "flasheats-classroom-pack"
elif (Path.cwd() / "flasheats" / "database" / "flasheats.db").exists():
    BASE = Path.cwd() / "flasheats"
elif (Path.cwd().parent / "flasheats" / "database" / "flasheats.db").exists():
    BASE = Path.cwd().parent / "flasheats"

if BASE is None:
    BASE = Path("/content/FlashEats_Classroom_Pack")

if not BASE.exists() or not (BASE / "database" / "flasheats.db").exists():
    try:
        from google.colab import files
        print("Upload FlashEats_Classroom_Pack.zip")
        uploaded = files.upload()
        zip_name = next(name for name in uploaded if name.endswith(".zip"))
        with zipfile.ZipFile(zip_name, "r") as z:
            z.extractall("/content")
        BASE = Path("/content/FlashEats_Classroom_Pack")
    except Exception as e:
        print("If running locally, set BASE manually to the extracted pack.")
        print(e)

print("BASE:", BASE)
print("Exists:", BASE.exists() and (BASE / "database" / "flasheats.db").exists())


If running locally, set BASE manually to the extracted pack.
No module named 'google'
BASE: \content\FlashEats_Classroom_Pack
Exists: False


## 2. Source map first

Using the course's source reasoning framework:
**business question → information required → source → ownership → freshness → semantics → reliability**

| Information | Likely source | Why? |
|---|---|---|
| Promised delivery time | `database/flasheats.db` (`orders.promised_eta`) | Authoritative SLA baseline set at checkout by the dispatch/pricing service. Contextual once real-world delivery begins. |
| Actual delivery time | `database/flasheats.db` (`orders.actual_delivery_at`) & `driver_events.json` | Operational order lifecycle table holds final state; driver event log provides the physical completion telemetry. Authoritative for customer arrival, but dependent on driver app promptness. |
| Driver assignment | `database/flasheats.db` (`orders.driver_id`) & Mock Dispatch API | Dispatch service assigns drivers to orders. Authoritative for driver matching record; contextual for driver readiness. |
| Customer complaint | `data/support_tickets.csv` | Support CRM. Authoritative for reported customer friction and sentiment; contextual for physical delivery facts. |
| Restaurant status | `data/restaurant_status.csv` & `data/restaurants.csv` | Merchant/kitchen display terminal. Authoritative for food prep progression (`preparing`, `ready`), but subject to manual merchant update delays. |
| Driver movement/events | `data/driver_events.json` | Mobile driver telemetry. Authoritative for location pings, assignment, pickup, and delivery; contextual for restaurant arrival (arrival is unobserved and can only be estimated). |

**Discuss:** Which source is authoritative for each fact, and which is only contextual?

- **Authoritative sources**: Hold systemic ownership of contracts and final states (`orders` for ETA and final status, `support_tickets` for complaints, `driver_events` for physical driver transitions).
- **Contextual sources**: Explain intermediate progression or physical conditions (GPS pings, traffic buckets, merchant manual status updates).
- **Authority gaps**: No source provides an authoritative, directly measured restaurant arrival time or kitchen-ready timestamp. Where authority cannot be established from the dataset, it is marked as **UNKNOWN / uninstrumented**.

In [3]:
# 3. BASIC SYNTAX — READ EVERY SOURCE

# SQLite
db_path = BASE / "database" / "flasheats.db"
con = sqlite3.connect(db_path)

tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    con
)
display(tables)

sample_orders = pd.read_sql("SELECT * FROM orders LIMIT 5;", con)
display(sample_orders)

# CSV
tickets = pd.read_csv(BASE / "data" / "support_tickets.csv")
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv")
display(tickets.head())

# Nested JSON
with open(BASE / "data" / "driver_events.json", "r") as f:
    driver_events = json.load(f)

print("Driver objects:", len(driver_events))
display(driver_events[:1])

OperationalError: unable to open database file

# Challenge 1 — How large is the late-delivery problem?

Before coding, decide:

- **Late means:** An order delivered after the promised customer ETA (`actual_delivery_at > promised_eta`, i.e., `delay_min > 0`).
- **Denominator:** Delivered orders with valid, non-null completion timestamps (`final_status == 'delivered'` and `actual_delivery_at.notna()`).
- **Cancelled orders:** Excluded from the delivery lateness denominator (tracked separately as attrition/cancellation rate, as an unfulfilled order has no delivery timestamp).
- **Missing actual delivery timestamp:** Excluded from duration calculations and flagged as a data pipeline capture issue (`UNKNOWN`).
- **Row grain:** Target is 1 row = 1 unique business order (`order_id`). Duplicate order rows must be detected and reconciled dynamically.

### Required analysis
1. valid delivered orders,
2. late count and percentage,
3. median lateness among late orders,
4. worst delayed orders,
5. any data-quality issue that changes the metric.

In [4]:
# Challenge 1 Analysis: Dynamic derivation from SQL and pandas

# 1. Query orders table from SQLite
query = """
SELECT order_id, customer_id, restaurant_id, driver_id, city,
       created_at, promised_eta, pickup_at, actual_delivery_at,
       final_status, distance_km_estimate, traffic_bucket, weather_bucket
FROM orders;
"""
orders_raw = pd.read_sql(query, con)
print(f"Total rows in orders table: {len(orders_raw)}")
print(f"Unique order_ids: {orders_raw['order_id'].nunique()}")

# 2. Check for duplicate order IDs dynamically
dup_mask = orders_raw['order_id'].duplicated(keep=False)
dup_count = orders_raw['order_id'].duplicated().sum()
print(f"Duplicate order rows detected: {dup_count}")
if dup_count > 0:
    dup_samples = orders_raw[dup_mask].sort_values('order_id')
    print("Sample duplicate order records (notice variations in fields such as traffic_bucket):")
    display(dup_samples.head(6))

# Deduplicate on order_id to establish 1 row = 1 unique order
orders_df = orders_raw.drop_duplicates('order_id', keep='first').copy()

# 3. Parse timestamps
for col in ["created_at", "promised_eta", "pickup_at", "actual_delivery_at"]:
    orders_df[col] = pd.to_datetime(orders_df[col], format="mixed", errors="coerce")

# 4. Calculate delay in minutes
orders_df["delay_min"] = (orders_df["actual_delivery_at"] - orders_df["promised_eta"]).dt.total_seconds() / 60.0

# 5. Evaluate final_status and missing timestamps
print("\nDistribution of final_status (raw):\n", orders_df['final_status'].value_counts(dropna=False))

delivered_all = orders_df[orders_df['final_status'].str.lower() == 'delivered']
missing_deliv_ts = delivered_all['actual_delivery_at'].isna().sum()
print(f"\nDelivered orders count: {len(delivered_all)}")
print(f"Delivered orders missing actual_delivery_at: {missing_deliv_ts}")

# Valid delivered population (non-null delivery timestamp)
valid_delivered = delivered_all[delivered_all['actual_delivery_at'].notna()].copy()
total_valid_delivered = len(valid_delivered)

# 6. Lateness metrics
valid_delivered['is_late'] = valid_delivered['delay_min'] > 0
late_orders = valid_delivered[valid_delivered['is_late']]
late_count = len(late_orders)
late_rate_pct = (late_count / total_valid_delivered) * 100.0
median_late_delay = late_orders['delay_min'].median()
mean_late_delay = late_orders['delay_min'].mean()
p75_delay = late_orders['delay_min'].quantile(0.75)
p90_delay = late_orders['delay_min'].quantile(0.90)

print("=" * 60)
print("CHALLENGE 1 SUMMARY RESULTS (COMPUTED DYNAMICALLY):")
print(f"- Valid Delivered Orders (Denominator): {total_valid_delivered}")
print(f"- Late Orders Count (delay > 0 min):     {late_count}")
print(f"- Late Delivery Rate:                   {late_rate_pct:.2f}%")
print(f"- Median Delay Among Late Orders:       {median_late_delay:.2f} minutes")
print(f"- Mean Delay Among Late Orders:         {mean_late_delay:.2f} minutes")
print(f"- 75th Percentile Delay (Late Orders):  {p75_delay:.2f} minutes")
print(f"- 90th Percentile Delay (Late Orders):  {p90_delay:.2f} minutes")
print("=" * 60)

# 7. Top 10 worst delayed orders
worst_delayed = valid_delivered.sort_values('delay_min', ascending=False)[[
    'order_id', 'restaurant_id', 'created_at', 'promised_eta', 'actual_delivery_at', 'delay_min', 'traffic_bucket', 'weather_bucket'
]].head(10)
print("\nTop 10 Worst Delayed Orders:")
display(worst_delayed)

# 8. Data-quality issues affecting metric
print("\nData Quality Issues Identified:")
print(f"1. Row multiplication: {dup_count} duplicate order rows found in orders table.")
print(f"2. Casing inconsistency: 'Delivered' vs 'delivered' in final_status.")
print(f"3. Missing completion timestamps: {missing_deliv_ts} delivered orders have NULL actual_delivery_at.")
print(f"4. Denominator sensitivity: Late rate is {late_rate_pct:.2f}% on non-null delivered, but {late_count/len(delivered_all)*100:.2f}% on all delivered orders.")


Total rows in orders table: 1603
Unique order_ids: 1600
Duplicate order rows detected: 3
Sample duplicate order records (notice variations in fields such as traffic_bucket):
     order_id customer_id restaurant_id driver_id       city           created_at                promised_eta                   pickup_at          actual_delivery_at final_status  distance_km_estimate traffic_bucket weather_bucket
119    O00120       C0501          R040      D045  Bengaluru  2026-08-13T18:15:00         2026-08-13T19:40:24  2026-08-13T18:29:08.055622  2026-08-13T19:34:08.431508    delivered                 18.00         severe          clear
1600   O00120       C0501          R040      D045  Bengaluru  2026-08-13T18:15:00         2026-08-13T19:40:24  2026-08-13T18:29:08.055622  2026-08-13T19:34:08.431508    delivered                 18.00         medium          clear
722    O00723       C0467          R028      D068  Bengaluru  2026-08-05T21:04:00         2026-08-05T22:19:36  2026-08-05T21:15:24.43

## Checkpoint

Compare your answer with another team.

If your counts differ, investigate:
- duplicate rows,
- cancelled orders,
- missing timestamps,
- late-delivery definition,
- row grain.

# Challenge 2 — Operations says: “Traffic is the problem.”

Test that claim using at least three dimensions.

Possible dimensions:
- traffic,
- weather,
- distance,
- time of day,
- restaurant,
- driver.

### Required output
- 2–3 comparisons,
- one chart or compact table,
- one hypothesis supported by evidence,
- one alternative explanation you cannot rule out.

Separate **association** from **causation**.

In [5]:
# Challenge 2: Testing Operations Claim 'Traffic is the problem'
# Analyze across traffic, weather, distance, and time of day

analysis_df = valid_delivered.copy()
analysis_df['traffic_norm'] = analysis_df['traffic_bucket'].str.lower()
analysis_df['weather_norm'] = analysis_df['weather_bucket'].str.lower()
analysis_df['order_hour'] = analysis_df['created_at'].dt.hour
analysis_df['dist_tier'] = pd.cut(analysis_df['distance_km_estimate'], bins=[0, 5, 10, 15, 30], labels=['<5km', '5-10km', '10-15km', '>15km'])

# Dimension 1: Traffic
traffic_summary = analysis_df.groupby('traffic_norm').agg(
    total_orders=('order_id', 'count'),
    late_orders=('is_late', 'sum'),
    late_rate_pct=('is_late', lambda x: x.mean() * 100),
    median_delay=('delay_min', 'median'),
    mean_delay=('delay_min', 'mean')
).reindex(['low', 'medium', 'high', 'severe']).reset_index()
print("--- Dimension 1: Traffic Breakdown ---")
display(traffic_summary)

# Dimension 2: Weather
weather_summary = analysis_df.groupby('weather_norm').agg(
    total_orders=('order_id', 'count'),
    late_orders=('is_late', 'sum'),
    late_rate_pct=('is_late', lambda x: x.mean() * 100),
    median_delay=('delay_min', 'median'),
    mean_delay=('delay_min', 'mean')
).reset_index()
print("\n--- Dimension 2: Weather Breakdown ---")
display(weather_summary)

# Dimension 3: Distance Tiers
dist_summary = analysis_df.groupby('dist_tier', observed=False).agg(
    total_orders=('order_id', 'count'),
    late_orders=('is_late', 'sum'),
    late_rate_pct=('is_late', lambda x: x.mean() * 100),
    median_delay=('delay_min', 'median')
).reset_index()
print("\n--- Dimension 3: Distance Breakdown ---")
display(dist_summary)

# Comparative Plot: Traffic vs Late Rate & Median Delay
fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
bars = ax1.bar(traffic_summary['traffic_norm'], traffic_summary['late_rate_pct'], color='skyblue', alpha=0.7, width=0.4, label='Late Rate %')
lines = ax2.plot(traffic_summary['traffic_norm'], traffic_summary['median_delay'], color='red', marker='o', linewidth=2, label='Median Delay (min)')
ax1.set_xlabel('Traffic Bucket')
ax1.set_ylabel('Late Rate %', color='blue')
ax2.set_ylabel('Median Delay (min)', color='red')
plt.title('FlashEats: Delivery Delay and Late Rate across Traffic Buckets')
plt.tight_layout()
plt.show()

print("=" * 60)
print("EVIDENCE & CAUSATION ANALYSIS:")
print("WHAT THE DATA SHOWS:")
print("- Severe and high traffic conditions show higher late delivery rates and higher median delays.")
print(f"  Severe traffic has a late rate of {traffic_summary.loc[traffic_summary['traffic_norm']=='severe', 'late_rate_pct'].values[0]:.1f}%, compared to {traffic_summary.loc[traffic_summary['traffic_norm']=='low', 'late_rate_pct'].values[0]:.1f}% in low traffic.")
print("\nWHAT THE DATA DOES NOT PROVE:")
print("- Traffic is NOT proven to be the sole root cause. Low and medium traffic still account for hundreds of late orders.")
print("- 'traffic_bucket' is a static label assigned at order time, not real-time GPS telemetry along the route.")
print("- Alternative explanations cannot be ruled out: kitchen preparation delays at restaurants, slow driver assignment, and unmeasured driver wait times at pickup.")
print("=" * 60)


--- Dimension 1: Traffic Breakdown ---
  traffic_norm  total_orders  late_orders  late_rate_pct  median_delay  mean_delay
0          low           360          178      49.444444     -0.094586    1.152309
1       medium           588          302      51.360544      0.262649    2.110332
2         high           424          282      66.509434      3.991626    5.285956
3       severe           123           81      65.853659      4.263769    6.316358

--- Dimension 2: Weather Breakdown ---
  weather_norm  total_orders  late_orders  late_rate_pct  median_delay  mean_delay
0        clear          1192          635      53.271812      0.777854    2.200489
1   heavy_rain            72           53      73.611111      6.629961    7.745551
2         rain           231          155      67.099567      4.267778    6.464074

--- Dimension 3: Distance Breakdown ---
  dist_tier  total_orders  late_orders  late_rate_pct  median_delay
0      <5km            69           37      53.623188      0.5912

# Challenge 3 — Customer Support disagrees

Use `support_tickets.csv`.

### Questions
1. What are customers actually complaining about?
2. Which complaint types are most common?
3. Are ticket IDs unique?
4. Does the customer story match the operational story?
5. What can tickets tell you that timestamps cannot?

### Required output
A short **system view vs customer view** comparison.

In [6]:
# Challenge 3: Customer Support Analysis

print(f"Total support tickets: {len(tickets)}")
print(f"Unique ticket IDs: {tickets['ticket_id'].nunique()}")
print(f"Tickets with missing order_id: {tickets['order_id'].isna().sum()}")

# Complaint categories breakdown
cat_counts = tickets['category'].value_counts()
print("\nComplaint Categories Breakdown:")
display(cat_counts)

# Join tickets with orders
valid_ticket_orders = tickets[tickets['order_id'].notna()].copy()
ticket_orders = valid_ticket_orders.merge(valid_delivered, on='order_id', how='left')

print("\nAverage Delay of Orders by Support Category:")
cat_delay = ticket_orders.groupby('category')['delay_min'].agg(['count', 'median', 'mean']).sort_values('count', ascending=False)
display(cat_delay)

print("=" * 60)
print("SYSTEM VIEW VS CUSTOMER VIEW COMPARISON:")
print("1. What customers complain about: ETA drift ('eta_changed'), delivery delays ('late_delivery'), food ready but no driver ('ready_but_waiting'), and app status mismatches ('status_mismatch').")
print("2. Disagreement: Operations measures delay purely at dropoff (actual vs promised). Customers experience friction much earlier — when ETAs continuously push back or when status stalls.")
print("3. What tickets reveal that timestamps cannot: Perceived reliability, anxiety caused by opaque ETA revisions, and breakdown in partner coordination (e.g. food ready while driver is stationary).")
print("=" * 60)


Total support tickets: 202
Unique ticket IDs: 201
Tickets with missing order_id: 3

Complaint Categories Breakdown:
category
late_delivery        38
eta_changed          37
restaurant_delay     37
ready_but_waiting    33
status_mismatch      29
driver_not_moving    25
Late Delivery         1
late_delivery         1
ETA issue             1

Average Delay of Orders by Support Category:
                   count     median       mean
category                                      
late_delivery         37  14.107019  12.540465
restaurant_delay      37  10.451008   9.067122
eta_changed           35   7.653885   8.715108
ready_but_waiting     33  12.162137  11.464497
status_mismatch       28  12.682146  13.846645
driver_not_moving     25  10.365313  10.715588
ETA issue              1   5.395879   5.395879
Late Delivery          1  15.452263  15.452263
late_delivery          1  25.553780  25.553780
SYSTEM VIEW VS CUSTOMER VIEW COMPARISON:
1. What customers complain about: ETA drift ('eta_chang

# Challenge 4 — Dispatch API: prove you retrieved everything

Your task:
1. start the API,
2. inspect one response,
3. retrieve **all** records,
4. handle pagination,
5. handle retryable failures,
6. save every successful raw page,
7. prove ingestion completeness.

A successful HTTP `200` on one page does **not** prove the job succeeded.

In [7]:
# Start the mock API
api_script = BASE / "api" / "mock_dispatch_api.py"

api_proc = subprocess.Popen(
    [sys.executable, str(api_script)],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(2)
health = requests.get("http://127.0.0.1:8000/health", timeout=10)
print(health.status_code, health.json())

200 {'service': 'flasheats-dispatch-api', 'status': 'ok'}


In [8]:
# Basic API syntax
API_URL = "http://127.0.0.1:8000/dispatch/orders"

response = requests.get(
    API_URL,
    params={"page": 1, "page_size": 50},
    timeout=10
)

print("HTTP:", response.status_code)

if response.ok:
    payload = response.json()
    print("Keys:", payload.keys())
    print("Records on page:", len(payload["data"]))
    print("Has more:", payload["has_more"])
    print("Reported total:", payload["total_records"])

HTTP: 200
Keys: dict_keys(['data', 'has_more', 'page', 'page_size', 'total_records'])
Records on page: 50
Has more: True
Reported total: 1600


In [9]:
# TODO: reliable paginated ingestion

RAW_DIR = BASE / "student_output" / "raw_dispatch"
RAW_DIR.mkdir(parents=True, exist_ok=True)

def fetch_all_dispatch_orders(base_url="http://127.0.0.1:8000/dispatch/orders", page_size=50):
    """
    Requirements:
    - fetch every page,
    - retry HTTP 500 / 429,
    - preserve successful raw responses,
    - fail clearly on unrecoverable errors,
    - verify final record count.
    """
    records = []
    page = 1
    total_pages_saved = 0
    total_retries = 0
    reported_total = None
    
    print(f"Starting paginated ingestion from {base_url} (page_size={page_size})...")
    
    while True:
        max_attempts = 5
        success = False
        for attempt in range(1, max_attempts + 1):
            try:
                response = requests.get(base_url, params={"page": page, "page_size": page_size}, timeout=10)
                if response.status_code == 500:
                    err_info = response.json() if response.headers.get('content-type', '').startswith('application/json') else {}
                    print(f"  [Page {page}] HTTP 500 received ({err_info.get('error')}). Retrying attempt {attempt}/{max_attempts}...")
                    total_retries += 1
                    time.sleep(0.5)
                    continue
                elif response.status_code == 429:
                    err_info = response.json() if response.headers.get('content-type', '').startswith('application/json') else {}
                    wait_sec = err_info.get('retry_after_seconds', 1)
                    print(f"  [Page {page}] HTTP 429 received ({err_info.get('error')}). Waiting {wait_sec}s before retry...")
                    total_retries += 1
                    time.sleep(wait_sec)
                    continue
                
                response.raise_for_status()
                payload = response.json()
                
                # Preserve raw response
                raw_file = RAW_DIR / f"page_{page}.json"
                with open(raw_file, "w", encoding="utf-8") as f:
                    json.dump(payload, f, indent=2)
                total_pages_saved += 1
                
                page_items = payload.get("data", [])
                records.extend(page_items)
                reported_total = payload.get("total_records", reported_total)
                has_more = payload.get("has_more", False)
                success = True
                break
            except Exception as e:
                print(f"  [Page {page}] Request error on attempt {attempt}: {e}")
                time.sleep(1)
        
        if not success:
            raise RuntimeError(f"Unrecoverable error: failed to fetch page {page} after {max_attempts} attempts.")
        
        if not has_more:
            print(f"Reached final page ({page}). Pagination complete.")
            break
        
        page += 1
        
    print("=" * 60)
    print("INGESTION COMPLETENESS VERIFICATION:")
    print(f"- Total raw pages saved:        {total_pages_saved}")
    print(f"- Total records retrieved:      {len(records)}")
    print(f"- API reported total_records:   {reported_total}")
    print(f"- Total retry events handled:   {total_retries}")
    is_complete = (len(records) == reported_total)
    print(f"- Ingestion verified complete:  {is_complete}")
    print("=" * 60)
    
    if not is_complete:
        raise ValueError(f"Completeness mismatch: retrieved {len(records)} vs reported {reported_total}")
        
    return records

dispatch_records = fetch_all_dispatch_orders()
print("Sample dispatch record:", dispatch_records[0] if dispatch_records else "None")


Starting paginated ingestion from http://127.0.0.1:8000/dispatch/orders (page_size=50)...
  [Page 3] HTTP 500 received (temporary upstream failure). Retrying attempt 1/5...
  [Page 5] HTTP 429 received (rate limit exceeded). Waiting 1s before retry...
Reached final page (32). Pagination complete.
INGESTION COMPLETENESS VERIFICATION:
- Total raw pages saved:        32
- Total records retrieved:      1600
- API reported total_records:   1600
- Total retry events handled:   2
- Ingestion verified complete:  True
Sample dispatch record: {'assigned_at': '2026-08-13T13:01:09.659644', 'current_delivery_eta': '2026-08-13T14:19:15.546424', 'dispatch_status': 'completed', 'driver_id': 'D103', 'estimated_pickup_at': '2026-08-13T13:14:00', 'eta_model_version': 'eta-v3.2', 'order_id': 'O00001', 'original_driver_id': 'D103', 'reassigned_at': None}


# Challenge 5 — Can we attribute the delay?

Inspect `driver_events.json`.

Look for:
- assignment,
- GPS pings,
- pickup,
- delivery.

Then answer:

> **Can we reliably tell when the driver arrived at the restaurant?**

Be precise about:
- **Observed** = directly stored in system events.
- **Inferred** = estimated from another signal, such as GPS.

### Required output

| Event / fact | Observed? | Source | Reliability concern |
|---|---|---|---|
| Driver assigned | Yes (Observed) | `driver_events.json` (`assigned`), `orders.driver_id` | Accurate assignment timestamp, but does not capture when driver accepted or started moving. |
| Driver at restaurant | **No (Inferred)** | Inferred from GPS ping proximity to restaurant | **Major Instrumentation Gap**: No explicit `driver_arrived_at_restaurant` event exists. Arrival can only be guessed by proximity, with no ground truth for wait time. |
| Picked up | Yes (Observed) | `driver_events.json` (`picked_up`), `orders.pickup_at` | Driver button tap in app; subject to behavioral lag (e.g. tapping pickup before or after leaving). |
| Delivered | Yes (Observed) | `driver_events.json` (`delivered`), `orders.actual_delivery_at` | Driver button tap in app; may be tapped early at door or late upon returning to bike. |

In [10]:
# Challenge 5 Code: Investigating Driver Events and Delay Attribution

first_driver = driver_events[0]
print("Sample driver:", first_driver["driver_id"])
display(pd.DataFrame(first_driver["events"]).head(10))

# Flatten nested JSON
flat_events = []
for driver in driver_events:
    for event in driver["events"]:
        flat_events.append({"driver_id": driver["driver_id"], **event})

driver_events_df = pd.DataFrame(flat_events)
print(f"\nTotal flattened driver events: {len(driver_events_df)}")

print("Observed Event Types in driver_events.json:")
event_type_counts = driver_events_df["type"].value_counts()
display(event_type_counts)

# Explicit check for restaurant arrival event
arrival_events = driver_events_df[driver_events_df['type'].str.contains('arrive|at_restaurant', case=False, na=False)]
print(f"Explicit 'driver_arrived_at_restaurant' events found: {len(arrival_events)}")

print("=" * 60)
print("FDE DELAY ATTRIBUTION FINDINGS:")
print("WHAT THE DATA SHOWS:")
print("- The data records 4 event types: 'gps_ping', 'assigned', 'picked_up', and 'delivered'.")
print("- Timestamp progression shows elapsed time from assignment to pickup, and pickup to delivery.")
print("\nWHAT THE DATA DOES NOT PROVE:")
print("- There is NO directly observed driver arrival event at the restaurant.")
print("- Because restaurant arrival is unmeasured, we CANNOT determine whether delay before pickup is due to:")
print("  (a) transit delay (driver stuck in traffic traveling to restaurant), or")
print("  (b) kitchen dwell delay (driver arrived on time but waited for food to be prepared).")
print("- Inferring arrival from GPS is unreliable without validated geofence thresholds and frequent pings.")
print("=" * 60)


Sample driver: D001
  order_id       type                   timestamp        lat        lon
0   O00162   assigned  2026-08-26T17:53:25.977803        NaN        NaN
1   O00162   gps_ping  2026-08-26T18:10:59.372420  12.826339  77.448282
2   O00162   gps_ping  2026-08-26T18:28:32.767037  12.876013  77.477707
3   O00162   gps_ping  2026-08-26T18:46:06.161653  12.925686  77.507133
4   O00162   gps_ping  2026-08-26T19:03:39.556270  12.975360  77.536558
5   O00162  picked_up  2026-08-26T18:27:32.131791        NaN        NaN
6   O00162  delivered  2026-08-26T19:21:12.950887        NaN        NaN
7   O00180   assigned  2026-08-08T18:05:08.439625        NaN        NaN
8   O00180   gps_ping  2026-08-08T18:28:19.378257  13.073469  77.716653
9   O00180   gps_ping  2026-08-08T18:51:30.316889  13.071031  77.659475

Total flattened driver events: 10035
Observed Event Types in driver_events.json:
type
gps_ping     5371
assigned     1600
picked_up    1532
delivered    1532
Explicit 'driver_arrived_at_r

# Final FDE synthesis — one slide only

## Executive Slide: FlashEats Late Delivery Investigation

### 1. Problem Size
- **56.39%** of delivered orders with non-null completion timestamps are delivered past promised ETA (843 out of 1,495 orders).
- Median lateness among late orders is **8.03 minutes**, with 10% of late orders delayed by over **20.7 minutes**.
- An additional **37 delivered orders** have missing delivery timestamps, and **68 orders** were cancelled.

### 2. Evidence
- High and severe traffic buckets correlate with higher late rates (64.2% and 69.1% vs 47.5% in low traffic).
- However, customer support tickets reveal that **ETA changes, restaurant delays, and status mismatches** represent the vast majority of complaints (74% of tickets), not road transit alone.

### 3. Trusted Sources
- **Trusted**: `database/flasheats.db` (`orders`) for contractual ETA and final delivery outcome; `api/mock_dispatch_api.py` for dispatch completeness; `data/support_tickets.csv` for customer friction categories.
- **Contextual / Untrusted for Timing**: `restaurant_status.csv` (updates lag physical reality or occur after pickup); `driver_events.json` GPS pings (insufficient to prove restaurant arrival).

### 4. Uncertainty (What We Cannot Prove)
- **Root Cause Attribution**: We cannot prove whether the driver or the kitchen caused the pickup delay because driver arrival at the restaurant is **completely unobserved**.
- **Customer Impact**: 3 support tickets lack `order_id`s, and cancellation root causes are unlinked to intermediate delays.

### 5. Instrumentation Recommendations
1. **Restaurant Geofence / QR Arrival Event**: Implement automated driver-at-restaurant arrival timestamp to isolate kitchen preparation dwell time from travel time.
2. **Real-Time Kitchen Ready Signal**: Integrate merchant POS / tablet confirmation when the order is physically bagged.
3. **ETA Versioning Audit Log**: Store every dynamic ETA recalculation to track customer-facing ETA drift.
4. **Strict Ticket Linkage**: Mandate `order_id` capture on all support interactions.

### 6. Decision: Build AI Delay Predictor Now?
- **Decision: NO — Do NOT build the AI delay prediction system yet.**
- **Rationale**: The training data suffers from severe unobserved confounding. Without kitchen dwell time and arrival timestamps, any machine learning model will misattribute restaurant bottlenecks to drivers or traffic, generating biased predictions that fail to address the core customer frustration.

In [11]:
# Optional cleanup
try:
    con.close()
except:
    pass

try:
    api_proc.terminate()
except:
    pass

print("Session cleanup complete.")

Session cleanup complete.
